# MODULE B - Data Engineering
#### Objective: The objective of Module B is to design, implement, and validate a reproducible, scalable, and auditable data engineering pipeline that transforms raw IBM AMLSim transaction data into clean, structured, and analysis-ready datasets for coordinated fraud ring detection and investigation.The pipeline must reliably process approximately 100,000 accounts, 12.48 million transactions, and 17,052 alert records, while preserving transaction integrity, account relationships, temporal ordering, and the original synthetic fraud labels.The module will focus on five engineering components:
#### B1. Raw data ingestion 
#### B2. Schema validation 
#### B3. Data cleaning and quarantine 
#### B4. SQL analytical storage 
#### B5. Reproducible pipeline


# B1 -Raw Data Ingestion
#### Objective: Develop a memory-efficient ingestion process for reading large CSV files, preserving source data, standardizing input handling, and recording ingestion metadata.

In [37]:
#B1.1 Recursive raw dataset inventory
from pathlib import Path
from datetime import datetime, timezone
import csv
import stat
DATASET_ROOT = Path(r"C:\Users\maiti\CAREER_HQ\09_PROJECTS_PORTFOLIO\Coordinated Fraud Ring Detection and Investigation\dataset\raw\100Kvertices-10Medges")
OUTPUT_PATH = Path(r"C:\Users\maiti\CAREER_HQ\09_PROJECTS_PORTFOLIO\Coordinated Fraud Ring Detection and Investigation\reports\dataset_manifest.csv")

FIELDNAMES = [
    "relative_path",
    "file_name",
    "extension",
    "size_bytes",
    "last_modified_utc",
]
def validate_paths(root: Path, output: Path) -> None:
    root=root.resolve()
    output=output.resolve()
    if not root.exists():
        raise FileNotFoundError(f'Dataset root not exist: {root}')
    if not root.is_dir():
        raise NotADirectoryError(f'Dataset root is not a directory : {root}')
    if output.suffix.lower() != ".csv" :
        raise ValueError("Output file must have a .csv extension")
    try:
       output.relative_to(root)
    except ValueError:
        pass
    else:
        raise ValueError("Output file cannot be inside the dataset root")
import os
from pathlib import Path

def discover_files(root: Path) -> list[Path]:
    files = []

    def handle_error(error):
        raise OSError(
            f"Failed to enumerate directory: {error.filename}"
        ) from error

    for dirpath, dirnames, filenames in os.walk(
        root,
        topdown=True,
        onerror=handle_error,
        followlinks=False
    ):
        current_dir = Path(dirpath)

        for filename in filenames:
            file_path = current_dir / filename

            try:
                file_stat = file_path.lstat()
            except OSError as e:
                raise OSError(f"Failed to retrieve metadata for {file_path}") from e

            if stat.S_ISLNK(file_stat.st_mode):
                continue

            if stat.S_ISREG(file_stat.st_mode):
                files.append(file_path)

    files.sort(
        key=lambda file: file.relative_to(root).as_posix()
    )

    return files
    

    
def extract_metadata(file_path: Path, root: Path) -> dict:
    #get file metadata once
    try:
        file_stats=file_path.stat()
    except OSError as e:
        raise OSError(f"failed to retrieve metadata for {file_path}") from e
    #POSIX - style relative path
    relative_path=file_path.relative_to(root).as_posix()
    #file name and extension
    file_name=file_path.name
    extension=file_path.suffix
    #file size in bytes
    size_bytes=file_stats.st_size
    # UTC ISO 8601 timestamp
    last_modified_utc = datetime.fromtimestamp(
        file_stats.st_mtime,
        tz=timezone.utc
    ).isoformat(timespec="microseconds")

    # Return metadata dictionary
    return {
        "relative_path": relative_path,
        "file_name": file_name,
        "extension": extension,
        "size_bytes": size_bytes,
        "last_modified_utc": last_modified_utc
    }
def build_manifest(root: Path) -> list[dict]:
    rows = []

    # Iterate through all discovered files
    for file_path in discover_files(root):
        rows.append(extract_metadata(file_path, root))

    return rows
def write_manifest(rows: list[dict], output: Path) -> None:
    # create parent directory if needed
    output.parent.mkdir(parents=True, exist_ok=True)
    with output.open("w",encoding="utf-8",newline="") as file:
        #create CSV writer
        writer=csv.DictWriter(file,fieldnames=FIELDNAMES,lineterminator="\n")
        #write header
        writer.writeheader()
        #write all rows
        writer.writerows(rows)


    

In [38]:
#main execution cell of B1.1
validate_paths(DATASET_ROOT, OUTPUT_PATH)

rows = build_manifest(DATASET_ROOT)

write_manifest(rows, OUTPUT_PATH)

print(f"Manifest created: {OUTPUT_PATH}")
print(f"Files inventoried: {len(rows)}")

Manifest created: C:\Users\maiti\CAREER_HQ\09_PROJECTS_PORTFOLIO\Coordinated Fraud Ring Detection and Investigation\reports\dataset_manifest.csv
Files inventoried: 14


In [39]:
#repeatibility testing of B1.1
manifest1 = build_manifest(DATASET_ROOT)
manifest2 = build_manifest(DATASET_ROOT)

assert manifest1 == manifest2

print("Repeatability test passed")


Repeatability test passed


#### Failure Scenario Testing of section B1.1

In [40]:
#test 1 : missing dataset
try:
    validate_paths(
        Path("missing_folder"),
        OUTPUT_PATH
    )
except FileNotFoundError:
    print("PASS")

PASS


In [41]:
#test 2 : Output not csv
try:
    validate_paths(
        DATASET_ROOT,
        Path("report.txt")
    )
except ValueError:
    print("PASS")

PASS


In [42]:
#test 3 : Ouput inside dataset
try:
    validate_paths(
        DATASET_ROOT,
        DATASET_ROOT / "manifest.csv"
    )
except ValueError:
    print("PASS")

PASS


In [43]:
#test 4 : metadata failure
try:
    extract_metadata(
        Path("does_not_exist.csv"),
        DATASET_ROOT
    )
except OSError:
    print("PASS")

PASS


In [44]:
#test 4 : directory-scan failure
import os

original_walk = os.walk

def failing_walk(*args, **kwargs):
    kwargs["onerror"](
        OSError("Simulated directory scan failure")
    )
    return []

os.walk = failing_walk

try:
    discover_files(DATASET_ROOT)
except OSError:
    print("PASS: Directory-scan failure detected")
else:
    raise AssertionError(
        "FAIL: Expected OSError was not raised"
    )
finally:
    os.walk = original_walk

PASS: Directory-scan failure detected


In [45]:
#test - 5:discovery stage metadata failure
# TEST: Discovery-stage metadata failure

from pathlib import Path

original_lstat = Path.lstat

def failing_lstat(self):
    raise OSError(
        "Simulated metadata access failure"
    )

Path.lstat = failing_lstat

try:
    discover_files(DATASET_ROOT)
except OSError:
    print("PASS: Metadata failure detected")
else:
    raise AssertionError(
        "FAIL: Expected OSError was not raised"
    )
finally:
    Path.lstat = original_lstat



PASS: Metadata failure detected


In [9]:
# B1.2 - Header, Schema and Sample Inspection
# B1.2.1 - transactions.csv inspection

import csv
from pathlib import Path


def inspect_transaction_csv(csv_path):
    try:
        with open(csv_path, "r", encoding="utf-8", newline="") as file:
            reader = csv.reader(file)

            # Handle empty file
            try:
                header = next(reader)
            except StopIteration:
                print("CSV file is empty")
                return

            print("Header:")
            print(header)

            print("\nNumber of Columns:")
            print(len(header))

            print("\nFirst 3 Records:")

            for i in range(3):
                try:
                    row = next(reader)
                except StopIteration:
                    break

                print(f"\nRecord {i + 1}:")
                print(row)

                print("Field Count:")
                print(len(row))

                # Report field-count mismatch
                if len(row) != len(header):
                    print(
                        f"Observation: Header has {len(header)} fields "
                        f"but this record has {len(row)} fields"
                    )

    except FileNotFoundError:
        print(f"Error: File not found -> {csv_path}")

    except PermissionError:
        print(f"Error: Permission denied -> {csv_path}")

In [11]:
csv_path=r"C:\Users\maiti\CAREER_HQ\09_PROJECTS_PORTFOLIO\Coordinated Fraud Ring Detection and Investigation\dataset\raw\100Kvertices-10Medges\transactions.csv"
inspect_transaction_csv(csv_path)

Header:
['TX_ID', 'SENDER_ACCOUNT_ID', 'RECEIVER_ACCOUNT_ID', 'TX_TYPE', 'TX_AMOUNT', 'TIMESTAMP', 'IS_FRAUD', 'ALERT_ID']

Number of Columns:
8

First 3 Records:

Record 1:
['1', '5942', '92982', 'TRANSFER', '516.95', '0', 'False', '-1']
Field Count:
8

Record 2:
['2', '86700', '43995', 'TRANSFER', '197.88', '0', 'False', '-1']
Field Count:
8

Record 3:
['3', '86700', '95516', 'TRANSFER', '197.88', '0', 'False', '-1']
Field Count:
8


In [14]:
# B1.2 - Header, Schema and Sample Inspection
# B1.2.1 accounts.csv inspection
def inspect_accounts_csv(account_csv_path):
    try:
        with open(account_csv_path,"r",encoding='utf-8',newline="") as file:
            reader=csv.reader(file)
            try:
                header=next(reader)
            except StopIteration:
                print(f"file is empty:{account_csv_path}")
            print("Header:")
            print(header)
            print("\nNumber Of Columns:")
            print(len(header))
            for i in range(3):
                try:
                    row = next(reader)
                except StopIteration:
                    break

                print(f"\nRecord {i + 1}:")
                print(row)

                print("Field Count:")
                print(len(row))

                # Report field-count mismatch
                if len(row) != len(header):
                    print(
                        f"Observation: Header has {len(header)} fields "
                        f"but this record has {len(row)} fields"
                    )


    except FileNotFoundError:
        print(f"Error:File not found - {account_csv_path}")
    except PermissionError:
        print(f"Error:Permission denied - {account_csv_path}")       

In [15]:
account_csv_path=r"C:\Users\maiti\CAREER_HQ\09_PROJECTS_PORTFOLIO\Coordinated Fraud Ring Detection and Investigation\dataset\raw\100Kvertices-10Medges\accounts.csv"
inspect_accounts_csv(account_csv_path)


Header:
['ACCOUNT_ID', 'CUSTOMER_ID', 'INIT_BALANCE', 'COUNTRY', 'ACCOUNT_TYPE', 'IS_FRAUD', 'TX_BEHAVIOR_ID']

Number Of Columns:
7

Record 1:
['0', 'C_0', '184.44', 'US', 'I', 'false', '1']
Field Count:
7

Record 2:
['1', 'C_1', '175.80', 'US', 'I', 'false', '1']
Field Count:
7

Record 3:
['2', 'C_2', '142.06', 'US', 'I', 'false', '1']
Field Count:
7


In [16]:
#B1.2 - Header, Schema and Sample Inspection
#B1.2.3 - alerts.csv inspection
#Generalization of the previous inspection function
import csv
from pathlib import Path


def inspect_csv(csv_path):
    try:
        with open(csv_path, "r", encoding="utf-8", newline="") as file:
            reader = csv.reader(file)

            # Handle empty file
            try:
                header = next(reader)
            except StopIteration:
                print("CSV file is empty")
                return

            print("File:", csv_path)
            print("\nHeader:")
            print(header)

            print("\nNumber of Columns:")
            print(len(header))

            print("\nFirst 3 Records:")

            for i in range(3):
                try:
                    row = next(reader)
                except StopIteration:
                    break

                print(f"\nRecord {i + 1}:")
                print(row)

                print("Field Count:")
                print(len(row))

                # Report field-count mismatch
                if len(row) != len(header):
                    print(
                        f"Observation: Header has {len(header)} fields "
                        f"but this record has {len(row)} fields"
                    )

    except FileNotFoundError:
        print(f"Error: File not found -> {csv_path}")

    except PermissionError:
        print(f"Error: Permission denied -> {csv_path}")

In [18]:
csv_path=r"C:\Users\maiti\CAREER_HQ\09_PROJECTS_PORTFOLIO\Coordinated Fraud Ring Detection and Investigation\dataset\raw\100Kvertices-10Medges\alerts.csv"
inspect_csv(csv_path)

File: C:\Users\maiti\CAREER_HQ\09_PROJECTS_PORTFOLIO\Coordinated Fraud Ring Detection and Investigation\dataset\raw\100Kvertices-10Medges\alerts.csv

Header:
['ALERT_ID', 'ALERT_TYPE', 'IS_FRAUD', 'TX_ID', 'SENDER_ACCOUNT_ID', 'RECEIVER_ACCOUNT_ID', 'TX_TYPE', 'TX_AMOUNT', 'TIMESTAMP']

Number of Columns:
9

First 3 Records:

Record 1:
['2036', 'cycle', 'True', '188', '24830', '33880', 'TRANSFER', '12.4', '0']
Field Count:
9

Record 2:
['842', 'fan_in', 'True', '556', '45959', '91430', 'TRANSFER', '3.07', '0']
Field Count:
9

Record 3:
['3000', 'cycle', 'True', '1038', '25885', '22925', 'TRANSFER', '13.06', '0']
Field Count:
9


In [10]:
#B1.3 - Data Types and Memory Assessment
# B1.3.1 — Transaction.csv memory baseline
import pandas as pd
def memory_baseline(csv_path):
    try:
        
        df = pd.read_csv(
            csv_path,
            dtype=str,
            nrows=100000,
            keep_default_na=False
        )

        print("SAMPLE SIZE:")
        print(f"Rows: {len(df)}")
        print(f"Columns: {len(df.columns)}")

        print("\nCOLUMN NAMES")
        print(list(df.columns))

        print("\nOBSERVED DATA TYPES")
        print(df.dtypes)

        print("\nMEMORY USAGE PER COLUMN (bytes)")
        memory_per_column = df.memory_usage(deep=True)
        print(memory_per_column)

        total_memory_bytes = memory_per_column.sum()
        total_memory_mib = total_memory_bytes / (1024 * 1024)

        print("\nTOTAL MEMORY")
        print(f"Bytes: {total_memory_bytes:,}")
        print(f"MiB: {total_memory_mib:.2f}")

        avg_memory_per_row = (
            total_memory_bytes / len(df)
            if len(df) > 0
            else 0
        )

        print("\nAVERAGE MEMORY PER ROW")
        print(f"{avg_memory_per_row:.2f} bytes")

    except FileNotFoundError:
        print(f"File not found: {csv_path}")
        
    except pd.errors.EmptyDataError:
       print("CSV file is empty")
    

In [11]:
#memory baseline for transaction.csv
transaction_path=r"C:\Users\maiti\CAREER_HQ\09_PROJECTS_PORTFOLIO\Coordinated Fraud Ring Detection and Investigation\dataset\raw\100Kvertices-10Medges\transactions.csv"
csv_path=transaction_path
memory_baseline(csv_path)

SAMPLE SIZE:
Rows: 100000
Columns: 8

COLUMN NAMES
['TX_ID', 'SENDER_ACCOUNT_ID', 'RECEIVER_ACCOUNT_ID', 'TX_TYPE', 'TX_AMOUNT', 'TIMESTAMP', 'IS_FRAUD', 'ALERT_ID']

OBSERVED DATA TYPES
TX_ID                  object
SENDER_ACCOUNT_ID      object
RECEIVER_ACCOUNT_ID    object
TX_TYPE                object
TX_AMOUNT              object
TIMESTAMP              object
IS_FRAUD               object
ALERT_ID               object
dtype: object

MEMORY USAGE PER COLUMN (bytes)
Index                      132
TX_ID                  5388895
SENDER_ACCOUNT_ID      5388249
RECEIVER_ACCOUNT_ID    5388601
TX_TYPE                5700000
TX_AMOUNT              5442638
TIMESTAMP              5000000
IS_FRAUD               5399856
ALERT_ID               5100266
dtype: int64

TOTAL MEMORY
Bytes: 42,808,637
MiB: 40.83

AVERAGE MEMORY PER ROW
428.09 bytes


In [14]:
#memory baseline for accounts.csv
account_path=r"C:\Users\maiti\CAREER_HQ\09_PROJECTS_PORTFOLIO\Coordinated Fraud Ring Detection and Investigation\dataset\raw\100Kvertices-10Medges\accounts.csv"
csv_path=account_path
memory_baseline(csv_path)

SAMPLE SIZE:
Rows: 100000
Columns: 7

COLUMN NAMES
['ACCOUNT_ID', 'CUSTOMER_ID', 'INIT_BALANCE', 'COUNTRY', 'ACCOUNT_TYPE', 'IS_FRAUD', 'TX_BEHAVIOR_ID']

OBSERVED DATA TYPES
ACCOUNT_ID        object
CUSTOMER_ID       object
INIT_BALANCE      object
COUNTRY           object
ACCOUNT_TYPE      object
IS_FRAUD          object
TX_BEHAVIOR_ID    object
dtype: object

MEMORY USAGE PER COLUMN (bytes)
Index                 132
ACCOUNT_ID        5388890
CUSTOMER_ID       5588890
INIT_BALANCE      5500000
COUNTRY           5100000
ACCOUNT_TYPE      5000000
IS_FRAUD          5382909
TX_BEHAVIOR_ID    5000000
dtype: int64

TOTAL MEMORY
Bytes: 36,960,821
MiB: 35.25

AVERAGE MEMORY PER ROW
369.61 bytes


In [15]:
#memory baseline for alerts.csv
alert_path=r"C:\Users\maiti\CAREER_HQ\09_PROJECTS_PORTFOLIO\Coordinated Fraud Ring Detection and Investigation\dataset\raw\100Kvertices-10Medges\alerts.csv"
csv_path=alert_path
memory_baseline(csv_path)

SAMPLE SIZE:
Rows: 17052
Columns: 9

COLUMN NAMES
['ALERT_ID', 'ALERT_TYPE', 'IS_FRAUD', 'TX_ID', 'SENDER_ACCOUNT_ID', 'RECEIVER_ACCOUNT_ID', 'TX_TYPE', 'TX_AMOUNT', 'TIMESTAMP']

OBSERVED DATA TYPES
ALERT_ID               object
ALERT_TYPE             object
IS_FRAUD               object
TX_ID                  object
SENDER_ACCOUNT_ID      object
RECEIVER_ACCOUNT_ID    object
TX_TYPE                object
TX_AMOUNT              object
TIMESTAMP              object
dtype: object

MEMORY USAGE PER COLUMN (bytes)
Index                     132
ALERT_ID               899626
ALERT_TYPE             928347
IS_FRAUD               903756
TX_ID                  956895
SENDER_ACCOUNT_ID      918945
RECEIVER_ACCOUNT_ID    918941
TX_TYPE                971964
TX_AMOUNT              911436
TIMESTAMP              877257
dtype: int64

TOTAL MEMORY
Bytes: 8,287,299
MiB: 7.90

AVERAGE MEMORY PER ROW
486.00 bytes


In [ ]:
#B1.3 - Data Types and Memory Assessment
# B1.3.2 — Building a memory estimation utility
def estimate_memory(sample_rows,sample_memory_bytes,full_dataset_rows,observed_memory_bytes):
    if sample_rows <= 0:
        raise ValueError("sample_rows must be greater than zero")
    if observed_memory_bytes <= 0:
        raise ValueError(
            "observed_memory_bytes must be greater than zero"
        )
    if min(sample_memory_bytes, full_dataset_rows, observed_memory_bytes) < 0:
        raise ValueError("Inputs cannot be negative")

    avg_memory_per_row = sample_memory_bytes / sample_rows
    estimated_full_memory_bytes = avg_memory_per_row * full_dataset_rows
    estimated_full_memory_mib = estimated_full_memory_bytes / (1024 ** 2)
    absolute_difference = abs(
        estimated_full_memory_bytes - observed_memory_bytes
    )
    relative_percentage_error = (
        absolute_difference / observed_memory_bytes * 100
        if observed_memory_bytes != 0
        else 0.0
    )

    return {
        "average_memory_per_row_bytes": avg_memory_per_row,
        "estimated_full_memory_bytes": estimated_full_memory_bytes,
        "estimated_full_memory_mib": estimated_full_memory_mib,
        "absolute_difference_bytes": absolute_difference,
        "relative_percentage_error": relative_percentage_error,
    }


In [18]:
results = estimate_memory(
    sample_rows=100000,
    sample_memory_bytes=42808637,
    full_dataset_rows=12476012,
    observed_memory_bytes=5387774219
)

print(results)

{'average_memory_per_row_bytes': 428.08637, 'estimated_full_memory_bytes': 5340810689.15644, 'estimated_full_memory_mib': 5093.393983036461, 'absolute_difference_bytes': 46963529.84356022, 'relative_percentage_error': 0.8716684837672523}
